In [ ]:
# pip install --quiet --upgrade pymongo gpt4all sentence_transformers


Note: you may need to restart the kernel to use updated packages.


In [1]:
MONGODB_URI = ("mongodb://localhost:8000/?directConnection=true")

In [ ]:
import pandas as pd
import pymongo
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
import logging
import os
import json
import requests
import voyageai
from dotenv import load_dotenv

load_dotenv()

# Connect to your local MongoDB deployment
# client = MongoClient(MONGODB_URI)
mongodb_connection = os.getenv("MongoDB_Client")
client = MongoClient(mongodb_connection)

# Select the sample_airbnb.listingsAndReviews collection
## Will need to add Zendesk Later for more robust data pulling purposes
collection = client["zendesk_ticket"]["Zoho_Ticket"]

pipeline = [
    {
        '$project': {
            'email': 1, 
            'createdTime': 1, 
            'subject': 1, 
            'program': 1, 
            'content': 1, 
            'closedTime': 1
        }
    }, {
        '$match': {
            '$expr': {
                '$gte': [
                    {
                        '$toDate': '$createdTime' # Need to convert Date String to Date Object for comparison
                    }, {
                        '$dateSubtract': {
                            'startDate': '$$NOW', 
                            'unit': 'day', 
                            'amount': 10
                        }
                    }
                ]
            }
        }
    }
]

results = collection.aggregate(pipeline)
result_list = list(results)
emails = [item['content'] for item in result_list]
emails
# zoho_ticket = pd.DataFrame(result_list) # Convert into Pandas DataFrame



['Hello, My name is Kiley Chaplin, and I am a PaCE student in the Child Life program. I have completed 7/10 required courses for the program, and am signed up to take the last three when classes start in a few weeks. I am trying to fill out my Eligibility Assessment on the ACLP website, and am having trouble. On the drop-down menu, UCSB isn\'t listed as an option as being an "ACLP-Endorsed Academic Program." Also, the form is asking for the name and contact information of the Academic Director, which I do not have. Further, the form is asking for a Program Enrollment Letter. Am I completing the correct form? How do I receive an enrollment letter if I am still in-progress on 3 courses? Who do I need to contact about getting this information? Thank you in advance for your help! Sincerely, Kiley Chaplin User-Agent : Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36 Referer URL : https://help.professional.ucsb.edu/',
 'Hello! My

In [25]:
print(len(emails))

185


### Create Embedding Function with Voyage AI + Storing Embedding Results in Collection  

In [ ]:
from pymongo.operations import SearchIndexModel
import time
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone
# Load the embedding model (https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1)
load_dotenv()

model = "voyage-4" # Voyage AI
vo = voyageai.Client(
    api_key=os.getenv("VOYAGE_API_KEY")
)
# print(os.getenv("Voyage_API_KEY"))
# Define a function to generate embeddings
def get_embedding(data):
  embeddings = vo.embed(data, model = model).embeddings
  return embeddings[0]



# model = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')
# model.save(model_path)
# model = SentenceTransformer(model_path)

# Define function to generate embeddings
# def get_embedding(text):
#     return model.encode(text).tolist()

# Calculate the cutoff date (10 days ago)
cutoff_date = datetime.now(timezone.utc) - timedelta(days=10)

# Filters for only documents with a content field and without an embedding field, within last 10 days
filter = {'$and': [ 
    { 'content': { '$exists': True, "$nin": [ None, "" ] } }, 
    { 'embedding': { '$exists': False } },
    { 'createdTime': { '$gte': cutoff_date.isoformat() } }
]} 

# Creates embeddings for all matching documents (no limit) - Purpose is to store embeddings in the collection 
updated_doc_count = 0
for document in collection.find(filter):
    text = document['content']
    embedding = get_embedding(text)
    collection.update_one({ '_id': document['_id'] }, { "$set": { 'embedding': embedding } }, upsert=True)
    updated_doc_count += 1
    print(f"Processed {updated_doc_count} documents...")

print("Documents updated: {}".format(updated_doc_count))


Processed 1 documents...
Processed 2 documents...
Processed 3 documents...
Processed 4 documents...
Processed 5 documents...
Processed 6 documents...
Processed 7 documents...
Processed 8 documents...
Processed 9 documents...
Processed 10 documents...
Processed 11 documents...
Processed 12 documents...
Processed 13 documents...
Processed 14 documents...
Processed 15 documents...
Processed 16 documents...
Processed 17 documents...
Processed 18 documents...
Processed 19 documents...
Processed 20 documents...
Processed 21 documents...
Processed 22 documents...
Processed 23 documents...
Processed 24 documents...
Processed 25 documents...
Processed 26 documents...
Processed 27 documents...
Processed 28 documents...
Processed 29 documents...
Processed 30 documents...
Processed 31 documents...
Processed 32 documents...
Processed 33 documents...
Processed 34 documents...
Processed 35 documents...
Processed 36 documents...
Processed 37 documents...
Processed 38 documents...
Processed 39 document

### Initialize/Create MongoDB Vector Search Index Model

In [32]:
index_name = "vector_index"
search_index_model = SearchIndexModel(
  definition = {
    "fields": [
      {
        "type": "vector",
        "numDimensions": 1024,
        "path": "embedding",
        "similarity": "cosine"
      }
    ]
  },
  name = index_name,
  type = "vectorSearch"
)
collection.create_search_index(model=search_index_model)

print("Polling to check if the index is ready. This may take up to a minute.")
predicate=None
if predicate is None:
   predicate = lambda index: index.get("queryable") is True
while True:
   indices = list(collection.list_search_indexes(index_name))
   if len(indices) and predicate(indices[0]):
      break
   time.sleep(5)
print(index_name + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index is ready for querying.


### Create Query Results and execute MQL pipeline

In [78]:
def get_query_results(query):
  """Gets results from a vector search query."""
  
  collection = client["zendesk_ticket"]["Zoho_Ticket"]

  query_embedding = get_embedding(query)
  pipeline = [
      {
            "$vectorSearch": {
              "index": "vector_index",
              "queryVector": query_embedding,
              "path": "embedding",
              "exact": True,
              "limit": 5
            }
      }, 
        # {
        #     "$addFields": {
        #         "createdTimeDate": {"$toDate": "$createdTime"}
        #     }
        # },
        # {
        #     "$match": {
        #         "$expr": {
        #             "$gte": [
        #                 "$createdTimeDate",
        #                 {
        #                     "$dateSubtract": {
        #                         "startDate": "$$NOW",
        #                         "unit": "day",
        #                         "amount": 10
        #                     }
        #                 }
        #             ]
        #         }
        #     }
        # },
        {
            "$project": {
                "_id": 0,
                "email": 1,
                "createdTime": 1,
                "createdTimeDate": 1,
                "subject": 1,
                "program": 1,
                "content": 1,
                "closedTime": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
  ]
  results = list(collection.aggregate(pipeline))
  emails = [item['content'] for item in results]
  # print(len(emails))
  return emails
# Test the function with a sample query
import pprint
pprint.pprint(get_query_results("Summary"))
# pprint.pprint(len(get_query_results("All Emails")))

['-- Jonathan Engeln Lambinet-Lacson Marketing Specialist Help & FAQ | | | |',
 '-- Jonathan Engeln Lambinet-Lacson Marketing Specialist Help & FAQ | | | |',
 '-- Jonathan Engeln Lambinet-Lacson Marketing Specialist Help & FAQ | | | |',
 'Hi team, The latest International Programs (UIP & ICP) evaluation results '
 'are now available. You can access the full report here: View Evaluation '
 'Report @Paolo Gardinali will follow-up with some historical perspective. '
 'Thanks everyone, and as always, great work supporting these programs. '
 'Best,-- Jonathan Engeln Lambinet-Lacson Manager, Automation & AI Integration '
 'Help & FAQ\xa0|\xa0 |\xa0 |\xa0 |\xa0',
 'This is a test assign to Jonathan-- Jonathan Engeln Lambinet-Lacson '
 'Marketing Specialist Help & FAQ | | | |']


### Implement LLM SandBox API for LLM Integration

In [123]:
load_dotenv(override=True)

LLM_API_ENDPOINT = os.getenv("LLMSANDBOX_API_ENDPOINT")
LLM_API_KEY = os.getenv("LLMSANDBOX_API_KEY") # Need to Hide this

headers = {
    "x-api-key": LLM_API_KEY,
    "Content-Type": "application/json"
}

def llm_integration_chatbot(questions: str):

    documents = get_query_results(questions) # Generate Vector Search Queries
    prompt = f"""Use the following pieces of context to answer the question at the end. 
        {documents}
        Question: {questions}
    """ # Full Prompt
    
    body = {"message": 
        {"role": "user", 
         "content": [{"contentType": "text", "body": prompt.strip()}], 
         "model": "claude-v4.5-sonnet"}}
    try:
        r = requests.post(f"{LLM_API_ENDPOINT}/conversation", headers=headers, json=body)
        r.raise_for_status()
        data = r.json()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
        print(f"Response body: {r.text}")
        return f"API Error: {e}"
    except Exception as e:
        print(f"Other error: {e}")
        return f"Error: {e}"

    if data:
        url = f"{LLM_API_ENDPOINT}/conversation/{data['conversationId']}"
        print(url)
        try:
            response = requests.get(url, headers={"x-api-key": LLM_API_KEY})
            response.raise_for_status()
            data_result = response.json()

            # More robust way to get the response
            message_keys = list(data_result['messageMap'].keys())
            print(message_keys)

            if not message_keys:
                return "No messages found in conversation."

            # Try to get the assistant's response (usually the last message)
            # If there are multiple messages, get the last one (most recent)
            # If there's only one message, that might be the user's message, so return a default
            else:
                return data_result['messageMap'][message_keys[-1]]['content'][0]['body']
        except requests.exceptions.HTTPError as e:
            print(f"HTTP error: {e}")
            print(f"Response body: {r.text}")
            return f"API Error: {e}"
        except Exception as e:
            print(f"Other error: {e}")
            return f"Error: {e}"


result = llm_integration_chatbot("Give me a short summary of what happened with the tickets in the past 10 days")
print(result)


https://6bk1nzbyfe.execute-api.us-east-1.amazonaws.com/api/conversation/01KMQ9GPBVE7N87JBHWX9NBG3Q
[]
No messages found in conversation.


### Ignore Below, Testing Purposes

In [134]:
# LLM_API_ENDPOINT = "https://6bk1nzbyfe.execute-api.us-east-1.amazonaws.com/api"
# LLM_API_KEY = "9edWpO1cce40LO34wjTbA2qlR7ZqbnCA4DnFjD8u"
load_dotenv(override=True)
LLM_API_ENDPOINT = os.getenv("LLMSANDBOX_API_ENDPOINT")
LLM_API_KEY = os.getenv("LLMSANDBOX_API_KEY")

headers = {
    "x-api-key": LLM_API_KEY,
    "Content-Type": "application/json"
}

# Ask Questions and Create Conversations
prompt = f"""SYSTEM: Please Give me a short summary based on the given information. "USER": {get_query_results("Summarize all tickets for the last ten days")}."""

body = {"message": 
        {"role": "user", 
         "content": [{"contentType": "text", "body": prompt.strip()}], 
         "model": "claude-v4.5-sonnet"}}

try:
    r = requests.post(f"{LLM_API_ENDPOINT}/conversation", headers=headers, json=body)
    r.raise_for_status()
    data = r.json()
    print(data)
except requests.exceptions.HTTPError as e:
    print("HTTP error:", e)
    print("Response body:", r.text)
except Exception as e:
    print("Other error:", e)

{'conversationId': '01KMQ9VB8QDJGAXTHH04HCAGKE', 'messageId': '01KMQ9VBNHMTN09HWY9KG1TTN2'}


In [125]:
url = f"{LLM_API_ENDPOINT}/conversation/01KMQ98GSSVA4AKQ20Y6TG57YH"
response = requests.get(url, headers={"x-api-key": LLM_API_KEY})
data = response.json()
print(data)

{'id': '01KMQ98GSSVA4AKQ20Y6TG57YH', 'title': '', 'createTime': 1774602961.7215586, 'messageMap': {'system': {'role': 'system', 'content': [{'contentType': 'text', 'body': ''}], 'model': 'claude-v4.5-sonnet', 'children': ['01KMQ98H5QFC9NFP4KP2SCFT4E'], 'feedback': None, 'usedChunks': None, 'parent': None, 'thinkingLog': None}, '01KMQ98H5QFC9NFP4KP2SCFT4E': {'role': 'user', 'content': [{'contentType': 'text', 'body': 'Use the following pieces of context to answer the question at the end. \n        ["Dear Extension Office, Hope you\'re doing well. I am a GAPP student under opt. I’m writing to double-check the status of my SEVIS record. My last day of work was January 31st, so my unemployment period officially kicked off on February 1st. I just wanted to make sure everything looks correct on your end and that my unemployment days are being tracked accurately in the system. I want to stay well within that 90-day limit. Could you let me know if my record is up to date? Also, if there\'s any

In [135]:
url = f"{LLM_API_ENDPOINT}/conversation/{data['conversationId']}"
response = requests.get(url, headers={"x-api-key": LLM_API_KEY})
data = response.json()
print(data)

{'id': '01KMQ9VB8QDJGAXTHH04HCAGKE', 'title': '', 'createTime': 1774603578.647683, 'messageMap': {}, 'lastMessageId': '', 'botId': '01KMPEK33Q42342WQ84TQ1389X', 'shouldContinue': False, 'status': None}


In [136]:
data_index = list(data['messageMap'].keys())
data_index

[]

In [133]:
data_index = list(data['messageMap'].keys())
# print(data_index)
data['messageMap']["01KMQ9SHDCAYVZ9W1TFKBH8YF8"]['content'][0]['body']

'# Summary\n\n**Two Main Topics:**\n\n1. **International Programs Evaluation Results**\n   - Jonathan Engeln Lambinet-Lacson (Manager, Automation & AI Integration) announced that the latest UIP & ICP evaluation results are available\n   - A full report can be accessed via a provided link\n   - Paolo Gardinali will provide historical context as follow-up\n\n2. **SEVIS Record Status Inquiry**\n   - Yufei Zhao, a GAPP student on OPT, is requesting verification of their SEVIS record status\n   - Their last work day was January 31st, making February 1st the start of their unemployment period\n   - They want to confirm unemployment days are being tracked correctly to stay within the 90-day limit\n   - Requesting confirmation if any additional actions are needed'